# Triggery (Triggers)

**Trigger** je procedura uložená přímo v databázi, která se **automaticky spustí** jako reakce na určitou událost (`INSERT`, `UPDATE` nebo `DELETE`) na dané tabulce — bez ohledu na to, odkud operace přišla (Python, web, SQL klient…).

---

## Přehled pojmů

| Pojem | Popis |
|---|---|
| `CREATE TRIGGER` | Vytvoří trigger |
| `DROP TRIGGER` | Smaže trigger |
| `SHOW TRIGGERS` | Vypíše existující triggery |
| `BEFORE` | Spustí se **před** operací — lze změnit hodnoty nebo operaci odmítnout |
| `AFTER` | Spustí se **po** operaci — vhodné pro logování |
| `NEW.sloupec` | Nová hodnota (INSERT, UPDATE) |
| `OLD.sloupec` | Původní hodnota (UPDATE, DELETE) |

## Kdy použít trigger?

| Použití | Příklad |
|---|---|
| **Audit log** | Po každém UPDATE zapiš starý a nový stav do logovací tabulky |
| **Automatický výpočet** | Před INSERT automaticky vypočti slevu z ceny |
| **Validace dat** | Odmítni INSERT s zápornou cenou (`SIGNAL SQLSTATE`) |
| **Referenční akce** | Po DELETE záznamu archivuj data do archivu |

In [ ]:
! pip install mysql-connector-python

## Připojení a příprava tabulek

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# Příprava tabulek (smažeme staré verze)
mycursor.execute("DROP TABLE IF EXISTS automobily_log")
mycursor.execute("DROP TABLE IF EXISTS automobily")

mycursor.execute("""
    CREATE TABLE automobily (
        id INT PRIMARY KEY AUTO_INCREMENT,
        spz CHAR(7) NOT NULL UNIQUE,
        pocet_sedadel INT NOT NULL,
        max_rychlost INT,
        nosnost INT,
        nutna_kvalifikace VARCHAR(50) NOT NULL DEFAULT 'B'
    )
""")

mycursor.execute("""
    CREATE TABLE automobily_log (
        log_id INT PRIMARY KEY AUTO_INCREMENT,
        akce VARCHAR(10) NOT NULL,
        spz CHAR(7),
        stara_rychlost INT,
        nova_rychlost INT,
        cas TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")

mycursor.execute("""
    INSERT INTO automobily (spz, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace) VALUES
        ('1A11111', 4, 190, 3, 'B'),
        ('2B22222', 2, 220, 2, 'B'),
        ('3C33333', 5, 160, 5, 'B'),
        ('4D44444', 3, 250, 2, 'A')
""")
mydb.commit()
print("✅ Tabulky připraveny.")

---

## Syntaxe CREATE TRIGGER

```sql
CREATE TRIGGER nazev_triggeru
    {BEFORE | AFTER} {INSERT | UPDATE | DELETE}
    ON nazev_tabulky
    FOR EACH ROW
BEGIN
    -- tělo triggeru
END;
```

### Přístup k hodnotám v těle triggeru

| Výraz | Dostupný v | Popis |
|---|---|---|
| `NEW.sloupec` | INSERT, UPDATE | Nová (vkládaná/měněná) hodnota |
| `OLD.sloupec` | UPDATE, DELETE | Původní hodnota |

> **Poznámka pro Python:** Triggery s `BEGIN...END` posílejte jako jedinou SQL zprávu. Pro jednořádkové triggery `BEGIN/END` není nutný.

## AFTER INSERT — audit log

Po každém vložení záznamu automaticky zapíše řádek do logovací tabulky:

In [ ]:
mycursor.execute("DROP TRIGGER IF EXISTS after_auto_insert")

mycursor.execute("""
    CREATE TRIGGER after_auto_insert
    AFTER INSERT ON automobily
    FOR EACH ROW
    INSERT INTO automobily_log (akce, spz, nova_rychlost)
    VALUES ('INSERT', NEW.spz, NEW.max_rychlost)
""")
print("✅ Trigger 'after_auto_insert' vytvořen.")

# Test
mycursor.execute("INSERT INTO automobily (spz, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace) VALUES ('5E55555', 8, 120, 10, 'D')")
mydb.commit()

mycursor.execute("SELECT * FROM automobily_log")
print("\n📋 Log po INSERT:")
for row in mycursor.fetchall():
    print(row)

## AFTER UPDATE — sledování změn

Zaloguje každou změnu `max_rychlost` — uloží starou i novou hodnotu:

In [ ]:
mycursor.execute("DROP TRIGGER IF EXISTS after_auto_update")

mycursor.execute("""
    CREATE TRIGGER after_auto_update
    AFTER UPDATE ON automobily
    FOR EACH ROW
    INSERT INTO automobily_log (akce, spz, stara_rychlost, nova_rychlost)
    VALUES ('UPDATE', NEW.spz, OLD.max_rychlost, NEW.max_rychlost)
""")
print("✅ Trigger 'after_auto_update' vytvořen.")

mycursor.execute("UPDATE automobily SET max_rychlost = 210 WHERE spz = '1A11111'")
mycursor.execute("UPDATE automobily SET max_rychlost = 180 WHERE spz = '3C33333'")
mydb.commit()

mycursor.execute("SELECT * FROM automobily_log ORDER BY log_id")
print("\n📋 Log po UPDATE:")
for row in mycursor.fetchall():
    print(row)

## BEFORE INSERT — validace dat

`BEFORE` trigger se spustí **před** zápisem do databáze. Lze:
- **Změnit hodnotu** pomocí `SET NEW.sloupec = ...`
- **Odmítnout operaci** pomocí `SIGNAL SQLSTATE '45000' SET MESSAGE_TEXT = 'popis'`

```sql
IF NEW.sloupec < 0 THEN
    SIGNAL SQLSTATE '45000'
    SET MESSAGE_TEXT = 'Hodnota musí být kladná!';
END IF;
```

In [ ]:
mycursor.execute("DROP TRIGGER IF EXISTS before_auto_insert_check")

mycursor.execute("""
    CREATE TRIGGER before_auto_insert_check
    BEFORE INSERT ON automobily
    FOR EACH ROW
    BEGIN
        IF NEW.max_rychlost <= 0 THEN
            SIGNAL SQLSTATE '45000'
            SET MESSAGE_TEXT = 'Chyba: max_rychlost musí být kladná!';
        END IF;
    END
""")
print("✅ Trigger 'before_auto_insert_check' vytvořen.\n")

# Test 1: platný INSERT
try:
    mycursor.execute("INSERT INTO automobily (spz, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace) VALUES ('6F66666', 4, 150, 3, 'B')")
    mydb.commit()
    print("✅ Platný INSERT proběhl.")
except Exception as e:
    mydb.rollback()
    print(f"❌ {e}")

# Test 2: neplatný INSERT
try:
    mycursor.execute("INSERT INTO automobily (spz, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace) VALUES ('7G77777', 4, -10, 3, 'B')")
    mydb.commit()
    print("✅ INSERT proběhl.")
except Exception as e:
    mydb.rollback()
    print(f"❌ Trigger zachytil chybu: {e}")

## BEFORE UPDATE — automatická korekce

Trigger upraví hodnotu ještě před zápisem pomocí `SET NEW.sloupec = ...`:

In [ ]:
mycursor.execute("DROP TRIGGER IF EXISTS before_auto_update_clamp")

mycursor.execute("""
    CREATE TRIGGER before_auto_update_clamp
    BEFORE UPDATE ON automobily
    FOR EACH ROW
    BEGIN
        IF NEW.max_rychlost < 50 THEN
            SET NEW.max_rychlost = 50;
        ELSEIF NEW.max_rychlost > 300 THEN
            SET NEW.max_rychlost = 300;
        END IF;
    END
""")
print("✅ Trigger 'before_auto_update_clamp' vytvořen.")

# Pokus nastavit příliš vysokou rychlost
mycursor.execute("UPDATE automobily SET max_rychlost = 999 WHERE spz = '1A11111'")
mydb.commit()
mycursor.execute("SELECT spz, max_rychlost FROM automobily WHERE spz = '1A11111'")
spz, r = mycursor.fetchone()
print(f"Rychlost po UPDATE na 999: {r}  (trigger ořezal na 300)")

## AFTER DELETE — archivace smazaných dat

Po smazání záznamu automaticky uloží informaci o mazání do logu:

In [ ]:
mycursor.execute("DROP TRIGGER IF EXISTS after_auto_delete")

mycursor.execute("""
    CREATE TRIGGER after_auto_delete
    AFTER DELETE ON automobily
    FOR EACH ROW
    INSERT INTO automobily_log (akce, spz, stara_rychlost)
    VALUES ('DELETE', OLD.spz, OLD.max_rychlost)
""")
print("✅ Trigger 'after_auto_delete' vytvořen.")

mycursor.execute("DELETE FROM automobily WHERE spz = '6F66666'")
mydb.commit()

mycursor.execute("SELECT log_id, akce, spz, stara_rychlost, nova_rychlost, cas FROM automobily_log ORDER BY log_id")
print("\n📋 Kompletní log (INSERT + UPDATE + DELETE):")
print(f"{'log_id':<8} {'akce':<8} {'spz':<9} {'stará':<7} {'nová':<7} čas")
print("-" * 55)
for row in mycursor.fetchall():
    print(f"{row[0]:<8} {row[1]:<8} {str(row[2]):<9} {str(row[3]):<7} {str(row[4]):<7} {row[5]}")

## Správa triggerů

```sql
-- Výpis všech triggerů na konkrétní tabulce
SHOW TRIGGERS LIKE 'automobily';

-- Smazání triggeru
DROP TRIGGER IF EXISTS nazev_triggeru;
```

In [ ]:
mycursor.execute("SHOW TRIGGERS LIKE 'automobily'")
triggers = mycursor.fetchall()
print(f"Triggery na tabulce 'automobily': {len(triggers)}")
for t in triggers:
    print(f"  {t[0]:<35} {t[1]} {t[2]}")

## Cvičení 1 — Automatický výpočet

1. Vytvořte tabulku `produkty` (id INT PK AUTO_INCREMENT, nazev VARCHAR(50), cena DECIMAL(10,2), sleva DECIMAL(10,2) DEFAULT 0).
2. Vytvořte `BEFORE INSERT` trigger, který automaticky nastaví `sleva = cena * 0.1`.
3. Vložte 3 produkty **bez uvedení slevy** a ověřte, že se automaticky dopočítala.

<details>
<summary>🔎 Ukázka řešení (klikni pro zobrazení)</summary>

```python
mycursor.execute("DROP TABLE IF EXISTS produkty")
mycursor.execute("""CREATE TABLE produkty (
    id INT PRIMARY KEY AUTO_INCREMENT,
    nazev VARCHAR(50), cena DECIMAL(10,2), sleva DECIMAL(10,2) DEFAULT 0)""")

mycursor.execute("DROP TRIGGER IF EXISTS auto_sleva")
mycursor.execute("""
    CREATE TRIGGER auto_sleva
    BEFORE INSERT ON produkty
    FOR EACH ROW
    SET NEW.sleva = NEW.cena * 0.1
""")

mycursor.execute("INSERT INTO produkty (nazev, cena) VALUES ('Notebook', 25000), ('Myš', 500), ('Klávesnice', 800)")
mydb.commit()

mycursor.execute("SELECT * FROM produkty")
for row in mycursor.fetchall():
    print(row)
```

</details>

In [ ]:
# Cvičení 1: Váš kód sem 👇


## Cvičení 2 — Audit log platů

1. Vytvořte tabulku `zamestnanci` (id INT PK AUTO_INCREMENT, jmeno VARCHAR(50), plat DECIMAL(10,2)).
2. Vytvořte tabulku `plat_log` (log_id INT PK AUTO_INCREMENT, zamestnanec_id INT, stary_plat DECIMAL(10,2), novy_plat DECIMAL(10,2), zmeneno TIMESTAMP DEFAULT CURRENT_TIMESTAMP).
3. Vytvořte `AFTER UPDATE` trigger na `zamestnanci`, který při změně `plat` zapíše záznam do `plat_log` s `OLD.plat` a `NEW.plat`.
4. Vložte 3 zaměstnance, dvěma změňte plat a zobrazte log.
5. Smažte triggery, tabulky a odpojte se.

<details>
<summary>🔎 Ukázka řešení (klikni pro zobrazení)</summary>

```python
mycursor.execute("DROP TABLE IF EXISTS plat_log")
mycursor.execute("DROP TABLE IF EXISTS zamestnanci")
mycursor.execute("""CREATE TABLE zamestnanci (id INT PRIMARY KEY AUTO_INCREMENT, jmeno VARCHAR(50), plat DECIMAL(10,2))""")
mycursor.execute("""CREATE TABLE plat_log (log_id INT PRIMARY KEY AUTO_INCREMENT, zamestnanec_id INT, stary_plat DECIMAL(10,2), novy_plat DECIMAL(10,2), zmeneno TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")

mycursor.execute("DROP TRIGGER IF EXISTS log_platu")
mycursor.execute("""
    CREATE TRIGGER log_platu
    AFTER UPDATE ON zamestnanci
    FOR EACH ROW
    INSERT INTO plat_log (zamestnanec_id, stary_plat, novy_plat)
    VALUES (OLD.id, OLD.plat, NEW.plat)
""")

mycursor.execute("INSERT INTO zamestnanci (jmeno, plat) VALUES ('Novák', 45000), ('Dvořák', 52000), ('Horák', 38000)")
mydb.commit()
mycursor.execute("UPDATE zamestnanci SET plat = 48000 WHERE jmeno = 'Novák'")
mycursor.execute("UPDATE zamestnanci SET plat = 40000 WHERE jmeno = 'Horák'")
mydb.commit()

mycursor.execute("SELECT * FROM plat_log")
for row in mycursor.fetchall():
    print(row)
```

</details>

In [ ]:
# Cvičení 2: Váš kód sem 👇

# Úklid na konci:
# mycursor.execute("DROP TRIGGER IF EXISTS log_platu")
# mycursor.execute("DROP TABLE IF EXISTS plat_log")
# mycursor.execute("DROP TABLE IF EXISTS zamestnanci")
# mydb.commit()
# mydb.close()


---

## 📋 Shrnutí

| Trigger | Kdy se spustí | Typické použití |
|---|---|---|
| `BEFORE INSERT` | Před vložením | Validace, automatický výpočet (`SET NEW.x`) |
| `AFTER INSERT` | Po vložení | Audit log, odvozené záznamy |
| `BEFORE UPDATE` | Před změnou | Ořez hodnot, validace |
| `AFTER UPDATE` | Po změně | Log změn (`OLD` vs `NEW`) |
| `BEFORE DELETE` | Před smazáním | Validace, podmíněné odmítnutí |
| `AFTER DELETE` | Po smazání | Archivace, kaskádní akce |

### Trigger vs. aplikační logika

| Trigger | Python / aplikace |
|---|---|
| Platí pro **všechny** přístupy k DB | Jen pro danou aplikaci |
| Audit log, referenční pravidla | Validace formulářů, UI feedback |
| Automatické výpočty v DB | Složitá business logika |